---
title: "8. Code and agents"
subtitle: "Pair programming with an assistant, and the assistant as an agent"
description: "How the coding assistant became an agent, why reading code now matters more than writing it, how to use code as a creative medium, and what it costs to supervise a loop that works on your brief."
---

If you have asked a coding assistant for help this year, you have already worked with an agent. The assistant in the editor or the terminal no longer only proposes the next line. It reads your files, writes a change, runs the code, reads the error message, and tries again until the tests pass, and then it reports back. That loop is what an [AI agent](https://en.wikipedia.org/wiki/Intelligent_agent) is, and this chapter takes it apart twice: first as a way of programming, then as a way of working.

The chapter works at the practice and interface layers of [the five layers](intro.ipynb#five-layers). The models are the language models of chapter [4](language.ipynb), and nothing new happens at the model layer. What changes is the working arrangement. Code is the one medium where the output can be checked by running it, so the craft is less about steering a generator and more about supervising one. And once the assistant can act, the thing you write stops being a prompt and becomes a brief: a bounded description of a goal, with enough context to make the result checkable and enough constraint to make the work finishable.

The chapter is for everyone, whether or not you have programmed before. The aim is to leave you able to read a hundred lines you did not write, say what each part does, and notice when a piece of it is wrong. It is also to leave you able to decide what you would and would not hand to a loop.

```{admonition} Question
:class: question
Before you read on, write down in one sentence how you would know that a piece of code an assistant gave you is wrong, assuming it runs without an error message. Keep your answer and hold it against the chapter summary.
```

## The assistant in the editor

By 2026 writing code with an assistant is the ordinary way of working in most professional settings. There is no reliable census of how widespread it is, since the surveys that report figures are answered by people who chose to answer them. What is not in doubt is that the tooling has settled into four patterns, stable since about 2023 even though the products carrying them have not.

**Tab completion.** The assistant proposes the next line or block as you type, in grey text you accept with a keystroke. This is the oldest of the four, dating from 2021, and the most invisible, because you never asked for the suggestion.

**Chat in the editor.** A side panel that can see your open files and answer in plain language. You paste a function and ask what it does, or describe a change and ask how to make it. Nothing is applied until you apply it.

**Inline edit.** You select a region of code, describe the change in a sentence, and the assistant returns a diff you accept or reject. This is the pattern with the tightest feedback loop, because the unit of work is small enough to read in full.

**Agentic.** You describe a task and the assistant plans it, edits several files, runs the code, reads the error message, and tries again, reporting back when it thinks it is done. This is the newest of the four. It became usable during 2025, and it is the one where the most code passes without a human reading it. The second half of this chapter is about this pattern.

The four differ in one respect that matters more than any feature comparison: how much unreviewed code each one puts into your project per interaction. Tab completion adds a line, inline edit adds a paragraph, chat adds whatever you paste in, and an agentic run can add several files. Your ability to supervise does not scale at the same rate, which is the practical argument for starting where the units are small.

In [ ]:
from matplotlib.patches import FancyBboxPatch, Rectangle

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"
patterns = [
    ("Tab completion", 0.7, "a line"),
    ("Inline edit", 2.2, "a paragraph"),
    ("Chat in the editor", 4.2, "whatever you paste in"),
    ("Agentic", 7.6, "several files"),
]

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.text(0.4, 5.05, "Four patterns, ordered by the code they add before anybody reads it",
        fontsize=11, fontweight="bold", color=purple)

for i, (name, length, gloss) in enumerate(patterns):
    y = 3.85 - i * 0.9
    widest = i == len(patterns) - 1
    ax.add_patch(FancyBboxPatch((0.4, y), 3.2, 0.7,
                                boxstyle="round,pad=0.02,rounding_size=0.12",
                                facecolor="#fff0f3" if widest else "#f4f1fa",
                                edgecolor=rose if widest else purple, linewidth=1.8))
    ax.text(2.0, y + 0.35, name, ha="center", va="center", fontsize=11, color=navy)
    ax.add_patch(Rectangle((4.0, y + 0.17), length, 0.36,
                           facecolor=rose if widest else purple))
    ax.text(4.0 + length + 0.15, y + 0.35, gloss, va="center", fontsize=10,
            color="#444444")

ax.annotate("", xy=(11.6, 0.85), xytext=(4.0, 0.85),
            arrowprops=dict(arrowstyle="-|>", color="#8a8598", lw=1.8))
ax.text(4.0, 0.35, "your ability to supervise does not grow along this axis",
        fontsize=10, color=rose)

ax.set_xlim(0, 12)
ax.set_ylim(0.1, 5.4)
ax.axis("off")
plt.tight_layout()
plt.show()

*Figure: The four patterns of coding assistance, ordered by how much unreviewed code each one adds per interaction, from the single line of a tab completion to the several files an agentic run touches.*

For a beginner the most useful pattern is chat in the editor, driven by three verbs: explain, fix, and refactor. You write a draft, or paste an example, the assistant explains it, you ask for a change, you run it, you iterate. The assistant does not have to be a hosted service. Open-weight models of a few billion parameters run on a recent laptop, with no account and no code leaving the machine. They are slower and weaker than the hosted ones, and for a sketch of a hundred lines they are enough; the [tools](tools.md) page says how to run one.

```{important}
The assistant is fastest when you can read code. You do not need to be able to write it from scratch.
```

## Reading before writing

That last point inverts how programming has been taught for fifty years. The traditional order is to learn the syntax, write small programs, and gradually read larger ones. The order that works with an assistant is the reverse. You will be handed working code within your first ten minutes, long before you could have produced it. From that moment on, the binding constraint on what you can build is not how fast you can write. It is whether you can tell good code from code that merely runs.

Call this supervision, and notice that it is a reading skill. Everything you do to an assistant's output is an act of reading: deciding whether to accept it, spotting the case it did not handle, saying why one of two versions is better. A model can produce more code in a minute than you can write in a day, so the bottleneck moved, and it moved to the one place the model cannot help you, which is judging its own work.

The habit that turns this into learning is explain-back. Take a function the assistant wrote for you and explain it, in your own words and in writing, line by line, before you look at any explanation. Then ask the assistant to check your account and tell you what you missed. The order matters. If you read the explanation first you will find it convincing, because a fluent explanation of code is exactly what a language model is good at producing, and agreeing with it feels like understanding. Explain-back is uncomfortable for about a week and then becomes fast, and the gap between "I followed that" and "I can say that" is where the illusion of competence lives.

This is why the course grades reading and explaining alongside shipping. Your practice log asks what you understood and what you changed, not only what you produced. A sketch that runs beautifully and that you cannot account for is worth less here than a plainer one you can take apart. The first will fail in week 10, when it needs one more feature and nobody in the room knows where anything is. The professional version of the argument is that code is read far more often than it is written, and a codebase nobody can explain is a liability whoever or whatever produced it. Edwards describes the personal cost from the inside: a programmer who lets an agent run all day ends up reviewing rather than making, and reviewing without understanding is the more tiring of the two [@Edwards2026Burnout].

## A worked example in p5.js

[p5.js](https://p5js.org/) is a JavaScript library descended from [Processing](https://en.wikipedia.org/wiki/Processing_(programming_language)), designed for visual sketches and maintained by the Processing Foundation [@P5js]. Both are open-source [creative-coding](https://en.wikipedia.org/wiki/Creative_coding) environments with large teaching communities behind them. You can use p5.js directly in your browser at the [p5.js web editor](https://editor.p5js.org/) without installing anything, which is why this course uses it.

A canonical "hello world" sketch:

```javascript
function setup() {
  createCanvas(400, 400);
}

function draw() {
  background(20);
  noStroke();
  fill(255);
  circle(mouseX, mouseY, 40);
}
```

Two functions do all the work. `setup` runs once and makes the canvas. `draw` runs about sixty times a second, and everything in it happens again on every frame. That single fact explains most of what beginners find surprising about p5.js, including why the screen is repainted from scratch each time and why removing the `background` line leaves a trail.

Now a prompt, which you can send even if you have never seen JavaScript before:

> "Modify this p5.js sketch so that instead of a single circle, twenty circles follow the mouse with a trailing delay, and their colour cycles through hues over time."

A capable assistant will produce something close to this:

```javascript
let circles = [];
const N = 20;

function setup() {
  createCanvas(400, 400);
  colorMode(HSB, 360, 100, 100, 1);
  for (let i = 0; i < N; i++) circles.push({ x: 200, y: 200 });
}

function draw() {
  background(20);
  noStroke();
  for (let i = N - 1; i > 0; i--) {
    circles[i].x = circles[i - 1].x;
    circles[i].y = circles[i - 1].y;
  }
  circles[0].x = mouseX;
  circles[0].y = mouseY;

  for (let i = 0; i < N; i++) {
    fill((frameCount + i * 18) % 360, 70, 90);
    circle(circles[i].x, circles[i].y, 40 - i * 1.5);
  }
}
```

Before you run it, practise the reading. Two pieces of notation carry the loops. Writing `circles[i]` picks out the item at position `i` in a list, and the `%` sign is the remainder operator, which wraps a rising number back around to a fixed range. The first loop counts *backwards*, and it has to, because each circle copies the position of the one in front of it. Counting forwards would copy the new position instead of the old one. That single detail is the whole trailing effect. It is the kind of thing that produces a sketch which looks almost right, and no error message would ever have told you about it.

Then run it, break it, and put it back. Set `N` to 200. Delete the `background` line. Change `40 - i * 1.5` to `40 + i * 1.5` and work out why the trail now grows instead of tapering. Ask for changes in the same conversation: make the trail spring rather than follow linearly, add a glow, make it react to the microphone. The tweaking is where the learning is.

## How to talk to an assistant

Some habits pay off immediately, and they carry over unchanged when the assistant becomes an agent later in the chapter.

**Say what kind of code you want.** "Vanilla JavaScript, no frameworks." "Python with `numpy`." "p5.js running in the browser." Without that, the assistant defaults to whatever was most common in its training data, which for JavaScript means a framework you did not ask for and cannot install in a web editor.

**Show the smallest example that fails.** Do not paste the whole project. A short reproduction is easier for the model to reason about, for the same reason it is easier for a person, and cutting it down often finds the bug before you send it.

**Ask for an explanation before you ask for a fix.** "Explain this function line by line as if I have never seen JavaScript." An assistant that has explained its own code is easier to argue with, and you get the vocabulary you need for the next question.

**Ask for the failure cases.** "What input would break this?" is a better question than "is this correct?", because the second invites agreement and the first invites work.

**Ask for tests.** "Write three small tests for this function, including one that should fail." Tests catch the mistakes neither of you can see by reading, and they are what an agent will later run in its loop.

**Change one thing at a time.** The discipline from chapter [5](images.ipynb) applies here too. If you accept three suggestions and then run the sketch, you have learned nothing about which of them broke it.

**Re-anchor often.** Long conversations drift, and an assistant fifty messages deep is still carrying decisions you abandoned twenty messages ago. Start a new chat for a new task and paste in the current state of the code.

**Verify by running it.** Models are wrong more often in code than in prose, and code is the one medium in this book where being wrong is instantly and cheaply detectable.

## Generating what the code arranges

Beyond writing the code, a model can also generate what the code arranges. That is the second half of creative coding with AI, and it is where the previous chapters come back.

- **Sprites, characters, and textures** for a game or a scene, using chapter [7](spatial.ipynb).
- **Backgrounds, skies, and reference frames** for an interactive piece, using chapter [5](images.ipynb).
- **Sound effects and ambience** for events and buttons, using chapter [6](sound.ipynb).
- **Voices** for characters, narration, and tutorials, with the consent rules from chapter [3](co-creation-and-ethics.ipynb) attached.

A workable pipeline has four steps. Sketch the idea on paper, including what the piece is for. Generate placeholder assets with image and audio tools, deliberately rough. Wire them together in code with an assistant. Then iterate on each piece separately, keeping the wiring fixed while you improve the assets and the assets fixed while you improve the wiring. That last clause is the whole trick, the equivalent of prototyping a board game in cardboard: it lets you find out whether the idea is any good before anything is expensive. One warning: placeholder assets have a way of becoming final assets, because replacing them is work and the piece already runs. Decide at step one which parts are meant to be generated in the finished piece and which stand in for something you intend to make, shoot, or record yourself, and write that decision in your log.

## Live coding

Some code is written in front of an audience. [Live coding](https://en.wikipedia.org/wiki/Live_coding) means writing and changing a program while it runs, so that the sound or the image shifts as the text on the screen changes. The convention is to project the editor. The draft manifesto of the [TOPLAP](https://toplap.org/) collective, founded in 2004, puts the ethic in four words: "Show us your screens." Blackwell and colleagues collect the history, the tools, and the arguments in one open-access volume [@Blackwell2022LiveCoding].

The practice belongs here for three reasons. The programmer becomes a performer, so the writing is the show rather than a step towards it. The language becomes an instrument, judged on how quickly a musical thought reaches the loudspeakers. And an assistant raises a question the older scene never had to answer: what is left of a performance when something else writes the next line faster than a person can type.

[Glicol](https://glicol.org/) is the easiest place to begin, since it runs in a browser tab with nothing installed. The name is short for graph-oriented live coding language, and the syntax follows the signal path. A line names a node, `>>` sends its output onward, and the chain reads as an oscillator into a filter into the output. Language and audio engine are both written in Rust and compiled to WebAssembly, which is where the near-native performance comes from. Glicol grew out of Qichao Lan's doctoral work at RITMO, University of Oslo, and has been performed collaboratively over the web [@Lan2021Glicol].

In [ ]:
from matplotlib.patches import FancyBboxPatch

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.text(0.4, 4.35, "The line on the projected screen", fontsize=11,
        fontweight="bold", color=purple)
ax.add_patch(FancyBboxPatch((0.4, 3.4), 11.2, 0.75,
                            boxstyle="round,pad=0.02,rounding_size=0.1",
                            facecolor=navy, edgecolor=navy))
ax.text(0.8, 3.775, "o: saw 220 >> lpf 800 1.0 >> mul 0.4", va="center",
        fontsize=11, family="monospace", color="#e7e5e4")

ax.text(0.4, 2.75, "The signal chain it names", fontsize=11,
        fontweight="bold", color=purple)
ax.text(11.6, 2.75, "each name in the line is a node in the chain", ha="right",
        fontsize=10, color=rose)

nodes = [(0.4, "oscillator", "a sawtooth at 220 Hz", False),
         (4.4, "filter", "cutting above 800 Hz", False),
         (8.4, "output", "at four tenths of full gain", True)]
for x, name, gloss, outcome in nodes:
    ax.add_patch(FancyBboxPatch((x, 1.35), 2.9, 0.85,
                                boxstyle="round,pad=0.02,rounding_size=0.12",
                                facecolor="#fff0f3" if outcome else "#f4f1fa",
                                edgecolor=rose if outcome else purple, linewidth=1.8))
    ax.text(x + 1.45, 1.92, name, ha="center", va="center", fontsize=11, color=navy)
    ax.text(x + 1.45, 1.58, gloss, ha="center", va="center", fontsize=10, color="#444444")
for x1, x2 in [(3.3, 4.4), (7.3, 8.4)]:
    ax.annotate("", xy=(x2, 1.775), xytext=(x1, 1.775),
                arrowprops=dict(arrowstyle="-|>", color=navy, lw=1.8))

ax.text(6.0, 0.55, "The program keeps running while it is edited, "
        "so a change to the line changes the sound in place.",
        ha="center", fontsize=10, color=navy)

ax.set_xlim(0, 12)
ax.set_ylim(0.2, 4.7)
ax.axis("off")
plt.tight_layout()
plt.show()

*Figure: One line of code names a chain of nodes, an oscillator into a filter into an output, so the sound follows the reading order of the text.*

:::{iframe} https://www.youtube-nocookie.com/embed/JEMhMlRQN0A
:width: 100%
An introduction to Glicol, the browser-based live coding language that grew out of doctoral work at RITMO, University of Oslo.
:::

[Watch on YouTube](https://www.youtube.com/watch?v=JEMhMlRQN0A)

[synth.is](https://synth.is/), developed by Björn Þór Jónsson at RITMO, comes at the same material from the other end. Rather than writing a synthesis graph, you breed one. An evolutionary search proposes sound-producing networks, a listener keeps whatever is worth keeping, and the next generation grows from those choices [@Jonsson2015Synthis]. The project has since made the resulting family tree performable, so a live coder travels a lineage of discovered sounds rather than typing each one [@Jonsson2025LiveCodingLineage].

The wider scene is larger than these two. TidalCycles expresses rhythm as patterns of cycles, and Strudel brings the same notation to the browser. Sonic Pi was built for classrooms, Hydra live-codes visuals rather than sound, SuperCollider sits underneath a good deal of it, and the club events where people dance to it are called [algoraves](https://en.wikipedia.org/wiki/Algorave). Try it yourself: open Glicol, load an example, and change one number while it is playing. Then carry one tension into the lab. Liveness rests on an audience watching a person think at a keyboard, so an assistant that finishes the line before it is typed changes what the room is watching.

## Rules and evolution as creative material

Creative coding did not begin with assistants, and it did not begin with models. Its older practice is procedural: you write a small routine, run it a great many times, and watch what comes out of the parts interacting. Chapter [2](how-it-works.ipynb) named two families of that kind, rules and evolution, and set them beside learning. This is where you write one of each.

Four patterns cover most of the territory. [L-systems](https://en.wikipedia.org/wiki/L-system) rewrite a string of symbols over and over according to a few rules, then read the result as turtle-drawing commands, which is how a handful of lines produces a fern or a coastline [@Prusinkiewicz1990]. [Cellular automata](https://en.wikipedia.org/wiki/Cellular_automaton) update a grid from the states of neighbouring cells, of which [Conway's Game of Life](https://en.wikipedia.org/wiki/Conway%27s_Game_of_Life) is the famous instance. Noise fields use a smooth pseudo-random function, `noise()` in p5.js, to give organic drift to positions, colours, and flow lines. Recursive subdivision splits a rectangle, then splits the pieces, and keeps going until a stopping rule bites. The same four patterns work on words. A grammar that rewrites symbols can rewrite phrases just as easily, which is how generative poetry has been written since the 1950s, and a noise field can drive a line break as well as a brush stroke.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

axiom = "F"
rules = {"F": "FF+[+F-F-F]-[-F+F+F]"}
angle = np.radians(23)


def grow(string, times):
    for _ in range(times):
        string = "".join(rules.get(symbol, symbol) for symbol in string)
    return string


def draw(ax, string):
    x, y, heading = 0.0, 0.0, np.pi / 2
    stack, segments = [], []
    for symbol in string:
        if symbol == "F":
            nx, ny = x + np.cos(heading), y + np.sin(heading)
            segments.append(((x, y), (nx, ny)))
            x, y = nx, ny
        elif symbol == "+":
            heading += angle
        elif symbol == "-":
            heading -= angle
        elif symbol == "[":
            stack.append((x, y, heading))
        elif symbol == "]":
            x, y, heading = stack.pop()
    for (x1, y1), (x2, y2) in segments:
        ax.plot([x1, x2], [y1, y2], color=purple, lw=1.0, solid_capstyle="round")
    points = np.array([point for segment in segments for point in segment])
    span = max(np.ptp(points[:, 0]), np.ptp(points[:, 1]), 1.0) * 0.62
    ax.set_xlim(points[:, 0].mean() - span, points[:, 0].mean() + span)
    ax.set_ylim(points[:, 1].min() - 0.1 * span, points[:, 1].min() + 1.95 * span)
    return len(segments)


fig, axes = plt.subplots(1, 4, figsize=(10, 3.8))
for step, ax in enumerate(axes):
    string = grow(axiom, step)
    count = draw(ax, string)
    heading = "the starting symbol" if step == 0 else (
        "after one rewrite" if step == 1 else f"after {step} rewrites")
    ax.set_title(heading, fontweight="bold", color=purple, pad=8)
    words = ("1 symbol and 1 line" if count == 1
             else f"{len(string)} symbols and {count} lines")
    ax.set_xlabel(words, color="#444444", labelpad=6)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(False)

fig.text(0.5, 0.015, "one rule, F becomes FF+[+F-F-F]-[-F+F+F], applied to its own "
         "output", ha="center", fontsize=10, color=navy)
plt.tight_layout(rect=(0, 0.055, 1, 1))
plt.show()

*Figure: One rewriting rule applied to its own output three times, where the tree is nowhere in the rule and comes entirely out of the repetition.*

Here is the second pattern in twenty-odd lines. Each row of pixels is one generation, and every cell takes the exclusive-or of its two neighbours in the row above.

```javascript
let row = [];
const CELL = 4;

function setup() {
  createCanvas(400, 400);
  noStroke();
  background(255);
  for (let i = 0; i < width / CELL; i++) row.push(0);
  row[floor(row.length / 2)] = 1;
}

function draw() {
  const y = frameCount * CELL;
  if (y > height) return;
  for (let i = 0; i < row.length; i++) {
    fill(row[i] ? 20 : 255);
    rect(i * CELL, y, CELL, CELL);
  }
  const next = [];
  for (let i = 0; i < row.length; i++) {
    next[i] = row[(i - 1 + row.length) % row.length] ^ row[(i + 1) % row.length];
  }
  row = next;
}
```

Paste it into the web editor and watch a triangle assemble itself out of a rule that contains no triangle. Then change the two lines that matter. Seed the first row at random rather than with a single cell, and swap the exclusive-or for another combination of the two neighbours, or bring `row[i]` itself into the rule so that each cell reads three cells rather than two. Each rule gives a different texture, and none of them is more work to write than this one.

In [ ]:
import numpy as np
from matplotlib.patches import Rectangle

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

width, generations = 129, 64
rows = np.zeros((generations, width), dtype=int)
rows[0, width // 2] = 1
for step in range(1, generations):
    rows[step] = np.roll(rows[step - 1], 1) ^ np.roll(rows[step - 1], -1)

fig = plt.figure(figsize=(10, 4.2), layout="constrained")
fig.get_layout_engine().set(w_pad=0.1, h_pad=0.16)
grid = fig.add_gridspec(1, 2, width_ratios=[1, 1.3])

rule = fig.add_subplot(grid[0, 0])
patterns = [(1, 1, 1), (1, 1, 0), (1, 0, 1), (1, 0, 0),
            (0, 1, 1), (0, 1, 0), (0, 0, 1), (0, 0, 0)]
size, pitch, row_pitch = 0.55, 2.35, 1.75
for index, pattern in enumerate(patterns):
    column, row = index % 4, index // 4
    x, y = 0.35 + column * pitch, 1.55 - row * row_pitch
    for cell, value in enumerate(pattern):
        rule.add_patch(Rectangle((x + cell * size, y + 0.8), size, size,
                                 facecolor=purple if value else "white",
                                 edgecolor="#8f8a9c", linewidth=1.0))
    rule.annotate("", xy=(x + 1.5 * size, y + 0.62), xytext=(x + 1.5 * size, y + 0.78),
                  arrowprops=dict(arrowstyle="-|>", color="#8f8a9c", lw=1.2))
    result = pattern[0] ^ pattern[2]
    rule.add_patch(Rectangle((x + size, y), size, size,
                             facecolor=rose if result else "white",
                             edgecolor=rose, linewidth=1.4))
rule.text(0.35, -1.2, "the middle cell becomes the exclusive or\nof its two "
          "neighbours", ha="left", fontsize=10, color=navy, linespacing=1.5)
rule.set_xlim(0, 9.6)
rule.set_ylim(-1.8, 2.9)
rule.set_aspect("equal")
rule.set_anchor("N")
rule.axis("off")
rule.set_title("The rule, in all eight cases",
               fontweight="bold", color=purple, loc="left")

picture = fig.add_subplot(grid[0, 1])
picture.imshow(rows, cmap="Purples", interpolation="nearest", aspect="auto")
picture.set_xticks([])
picture.set_yticks([])
picture.set_xlabel("one row of pixels for each generation, running downwards",
                   color="#444444")
picture.set_title("What 64 generations draw",
                  fontweight="bold", color=purple, loc="left")
for side in picture.spines.values():
    side.set_visible(True)
    side.set_color("#cfc7dd")

plt.show()

*Figure: The rule that gives every cell the exclusive or of its two neighbours, and the triangle that 64 generations of it draw without any triangle being written anywhere.*

Rules are one half of the older practice and evolution is the other, and a sketch is a good place to feel it. Give your drawing a small genome: three colours, an angle, a branch count, a line weight. Draw nine variants in a grid, each one the genome with a small random change. The user clicks the variant they like, that genome becomes the parent, and nine fresh mutants are drawn from it. Four or five rounds in you have something you could not have specified in advance and did not draw by hand. That is [interactive evolution](https://en.wikipedia.org/wiki/Interactive_evolutionary_computation). [Karl Sims](https://en.wikipedia.org/wiki/Karl_Sims) put it on the SIGGRAPH stage in 1991 [@Sims1991], and Picbreeder ran the same loop in a browser in 2008 [@Secretan2008]. Try it yourself with the [Evolve a picture](https://fourms.github.io/Creative-AI/apps/evolve/) app before you write your own.

An assistant earns its place here. The rewriting engine, the neighbour arithmetic, the grid layout, and the mutation function are fiddly, well documented, and uninteresting to get right by hand, so ask for them. What no assistant can do is decide that the angle should be 22 degrees rather than 25, or that the third tile in the second row is the one worth breeding from. The code is delegable and the judgement is not, and in a rule-based or evolutionary sketch the judgement is the work.

## A tiny AI-powered web tool

By this week you should be able to assemble something like:

- a page where a visitor types a sentence and a generated image appears,
- a sketch that listens to the microphone and reacts in colour,
- a button that produces a short generated story riff on a theme,
- a small tool that sorts a folder of your own files into categories using a local model.

Each of these needs the same four parts, whatever the tool does.

**A front end**, meaning a page and one file of JavaScript. Keep it to one file for as long as you can. A build system is a thing you will otherwise spend the afternoon configuring instead of making.

**A model call**, which happens either in the browser or on a server. In the browser is simpler to deploy and limited to small models; on a server you can call any model, and you then have a key to look after.

**A key that is not in the page.** Anything shipped to the browser is public, including an API key sitting in your JavaScript. This is the single most common mistake in student projects, and it is expensive rather than merely embarrassing. The fix is to keep the key on a server or to use a model that runs locally. Ask your assistant to check for this explicitly, because it will happily write the insecure version if that is what you asked for.

**Somewhere to put it.** A free static host is enough for anything in this course, and several will also run a small model demo for you.

In [ ]:
from matplotlib.patches import FancyBboxPatch

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

fig, ax = plt.subplots(figsize=(10, 4.2))


def panel(x, y, width, height, title, lines, face="#f4f1fa", edge=None):
    ax.add_patch(FancyBboxPatch((x, y), width, height,
                                boxstyle="round,pad=0.02,rounding_size=0.14",
                                facecolor=face, edgecolor=edge or purple, linewidth=1.8))
    ax.text(x + 0.3, y + height - 0.32, title, ha="left", va="center", fontsize=11,
            fontweight="bold", color=navy)
    for index, line in enumerate(lines):
        ax.text(x + 0.3, y + height - 0.78 - index * 0.36, line, ha="left",
                va="center", fontsize=10, color="#444444")


panel(0.3, 1.5, 4.3, 2.1, "In the browser",
      ["a page and one JavaScript file", "what the visitor types and sees",
       "everything here is public"])
panel(7.1, 1.5, 4.3, 2.1, "On a small server",
      ["one function that forwards the call", "the API key lives here",
       "nothing here is shipped to the page"], "#fff0f3", rose)

ax.annotate("", xy=(7.05, 2.9), xytext=(4.65, 2.9),
            arrowprops=dict(arrowstyle="-|>", color=navy, lw=1.8))
ax.text(5.85, 3.05, "the request", ha="center", fontsize=10, color="#444444")
ax.annotate("", xy=(4.65, 2.3), xytext=(7.05, 2.3),
            arrowprops=dict(arrowstyle="-|>", color=navy, lw=1.8))
ax.text(5.85, 2.05, "the result", ha="center", fontsize=10, color="#444444")

ax.add_patch(FancyBboxPatch((7.1, 0.15), 4.3, 0.8,
                            boxstyle="round,pad=0.02,rounding_size=0.12",
                            facecolor="#fbfaff", edgecolor="#cfc7dd", linewidth=1.5))
ax.text(9.25, 0.55, "The model provider", ha="center", va="center", fontsize=11,
        color=navy)
ax.annotate("", xy=(9.25, 1.0), xytext=(9.25, 1.45),
            arrowprops=dict(arrowstyle="<|-|>", color=navy, lw=1.6))

ax.add_patch(FancyBboxPatch((0.3, 0.15), 4.3, 0.8,
                            boxstyle="round,pad=0.02,rounding_size=0.12",
                            facecolor="#fbfaff", edgecolor="#cfc7dd", linewidth=1.5))
ax.text(2.45, 0.55, "A free static host", ha="center", va="center", fontsize=11,
        color=navy)
ax.annotate("", xy=(2.45, 1.45), xytext=(2.45, 1.0),
            arrowprops=dict(arrowstyle="-|>", color=navy, lw=1.6))

ax.text(5.85, 4.05, "A key put in the page is a key you have published",
        ha="center", fontsize=11, fontweight="bold", color=rose)

ax.set_xlim(0, 11.7)
ax.set_ylim(0.0, 4.4)
ax.axis("off")
plt.tight_layout()
plt.show()

*Figure: The four parts of a small web tool, with the key kept on the server because everything shipped to the browser is public.*

An assistant can assemble all four in an afternoon. What it cannot do is decide what the tool is for, and a tool that does something oddly specific and useful to one person is always more interesting in an exposition than a general one that does nothing in particular.

:::{note} Dig deeper: what the tools are called
:class: dropdown
The body of this chapter describes categories rather than products, because the categories have been stable since 2023 and the products have not. For the record, and because you will meet the names the moment you start searching: tab completion is most associated with GitHub Copilot, widely credited with bringing the pattern to a mass audience from 2021. Cursor and Windsurf are editors built around chat and inline edit [@CursorDocs]. Claude Code and Codex CLI are agentic assistants that work from a terminal rather than an editor panel [@ClaudeCode], and Cursor's agent mode is the same pattern inside the editor. [2026?: check that these four are still the names students meet.]

Agent frameworks that let you build a loop yourself include OpenAI's [Agents SDK](https://openai.github.io/openai-agents-python/) and LangGraph. Both give you tool definitions, a loop, and a way to inspect the trace. The early open experiments that made the pattern visible were AutoGPT [@AutoGPT] and BabyAGI [@BabyAGI], both from 2023, and OpenAI's Operator was the first widely used browser agent [@OpenAIOperator]. [2026?: Operator was folded into the agent mode of ChatGPT during 2025; confirm the current name.]

For the tiny web tool, the stack most students end up with is a page and one JavaScript file, built with Vite or Bun if a build step is needed at all. Browser-side model calls go through transformers.js or WebLLM. Hosting is Vercel or Netlify for a static page, or Hugging Face Spaces if you want a demo with a model attached and no server of your own [@HuggingFaceSpaces].

Names in this note are here so that you can search effectively. Do not read the list as a recommendation, and expect at least one entry to be gone or renamed by the time you read it.
:::

## What an agent is

The agentic pattern at the top of the chapter is the general case, and it is worth being able to name its parts, because almost every product in this space is a variation on the same three.

1. **Tools.** A language model with things it can call: web search, code execution, a file system, a browser, an external API.
2. **A loop.** The model takes a step, observes the result, plans the next step, and repeats.
3. **A stopping criterion.** The goal is met, the user intervenes, or a budget is exhausted.

That is the whole of it. There is no separate planning module and no reasoning engine bolted on the side. The planning happens in the same next-token machinery from chapter [2](how-it-works.ipynb). What makes it look like planning is that the output of one pass through the model is fed back in as the input to the next, together with whatever the tool returned.

In [ ]:
from matplotlib.patches import FancyBboxPatch

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

fig, ax = plt.subplots(figsize=(10, 3.8))


def box(x, y, w, h, title, gloss=None, face="#f4f1fa", edge=None, colour=None):
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                                boxstyle="round,pad=0.02,rounding_size=0.15",
                                facecolor=face, edgecolor=edge or purple, linewidth=1.8))
    ax.text(x + w / 2, y + h / 2 + (0.18 if gloss else 0), title, ha="center",
            va="center", fontsize=11, fontweight="bold", color=colour or navy)
    if gloss:
        ax.text(x + w / 2, y + h / 2 - 0.2, gloss, ha="center", va="center",
                fontsize=10, color="#e6dff0" if colour == "white" else "#444444")


box(0.3, 3.0, 1.9, 0.95, "A goal")
box(2.9, 2.85, 2.9, 1.25, "Model", "plans the next step", purple, purple, "white")
box(6.6, 2.85, 2.9, 1.25, "Tool", "search, code, files, browser")
box(10.2, 3.0, 1.6, 0.95, "Stop", None, "#fff0f3", rose)
ax.text(11.0, 3.15, "goal or budget", ha="center", fontsize=10, color="#444444")

ax.annotate("", xy=(2.9, 3.475), xytext=(2.2, 3.475),
            arrowprops=dict(arrowstyle="-|>", color=navy, lw=1.8))
ax.annotate("", xy=(6.6, 3.85), xytext=(5.8, 3.85),
            arrowprops=dict(arrowstyle="-|>", color=navy, lw=1.8))
ax.annotate("", xy=(5.8, 3.15), xytext=(6.6, 3.15),
            arrowprops=dict(arrowstyle="-|>", color=navy, lw=1.8))
ax.annotate("", xy=(10.2, 3.475), xytext=(9.5, 3.475),
            arrowprops=dict(arrowstyle="-|>", color=navy, lw=1.8))
ax.text(6.2, 4.0, "call", ha="center", fontsize=10, color="#444444")
ax.text(6.2, 2.85, "result", ha="center", fontsize=10, color="#444444")

ax.add_patch(FancyBboxPatch((2.9, 0.9), 6.6, 0.95,
                            boxstyle="round,pad=0.02,rounding_size=0.15",
                            facecolor="#fbfaff", edgecolor="#cfc7dd", linewidth=1.6))
ax.text(6.2, 1.55, "The trace", ha="center", fontsize=11,
        fontweight="bold", color=purple)
ax.text(6.2, 1.2, "every decision, every call, and everything that came back",
        ha="center", fontsize=10, color="#444444")
for x in (4.35, 8.05):
    ax.plot([x, x], [2.85, 1.85], color="#b9b4c4", lw=1.2, ls="--")

ax.text(6.0, 0.4, "The trace is what you supervise, because the mistake is usually "
        "three steps before the failure.", ha="center", fontsize=10, color=navy)

ax.set_xlim(0, 12)
ax.set_ylim(0.1, 4.5)
ax.axis("off")
plt.tight_layout()
plt.show()

*Figure: A goal feeds a model that plans the next step, the model calls a tool and reads the result, and the loop ends at a stopping criterion, with every pass written into the trace.*

Agents are not new as an idea; research systems that pursued goals through action existed decades before generative models. What changed around 2023 is the quality of the component in the middle. Language models became good enough at writing a sensible next step and, more importantly, at noticing that the last step failed and trying something else. Error recovery is what turns a loop into an agent rather than into an infinite one. Salma and colleagues call the shift a move from treating a system as an executor of instructions to treating it as a collaborator in a process [@Salma2025]. An agent does not answer your prompt, it works on your brief, and the difference shows up in how much of the outcome you can predict when you press go.

Three categories have different risk profiles, and you will meet all three this week.

- **A coding agent** works inside a repository, edits files, runs tests, and reads the errors. Its outputs are cheap to check, because code either runs or does not, which is the argument of the first half of this chapter.
- **A research agent** plans a search, reads sources, and drafts a summary. Its outputs look finished long before they are correct.
- **A browser-using agent** clicks through interfaces on your behalf. It can act on the world, and the world does not have an undo button.

Whatever the category, the artefact an agent produces is not only the result. It is also the trace: the ordered record of what the agent decided, which tool it called, what came back, and what it did next. The trace is the thing you supervise. A student who reads traces learns more in a week about how these systems work than one who reads only outputs. The trace is where the loop is visible, and where the mistake usually happened three steps before the visible failure.

## A creative pipeline as an agent

Take a concrete case. You want to produce a sixty-second animated short. The plan an agent would work through looks roughly like this.

1. **Story.** Ask a language model for three outlines on the brief, and pick one (chapter [4](language.ipynb)).
2. **Storyboard.** Generate twelve panels with an image model (chapter [5](images.ipynb)).
3. **Animation.** Run image-to-video on each panel with motion prompts (chapter [5](images.ipynb)).
4. **Voice-over.** Generate the narration with a speech model (chapter [6](sound.ipynb)).
5. **Music.** Generate a score (chapter [6](sound.ipynb)).
6. **Sound effects.** Generate Foley and ambience.
7. **Assembly.** Place the clips on a timeline, sync the voice-over, mix the audio.
8. **Render.** Export the film.

Every one of those steps is doable today by a separate tool, and you have used most of them already in this course. What is new is that the wiring between them, the exporting and importing and renaming and re-prompting, is itself automatable. Stitching the steps together used to be a producer's job, and in 2026 that producer is increasingly a loop. This is not the same claim as "AI replaces a film crew". Steps 2 to 6 produce material whose quality is a matter of taste, and the agent has none. Step 7 produces a result whose quality is a matter of timing, and the agent cannot hear it. What the agent removes is the mechanical labour between the creative decisions, which is often most of the hours, but it is not the decisions.

Notice also what the numbered list hides. Written out as eight steps it looks linear, and it is not. Step 3 will produce a shot that contradicts the storyboard, which sends you back to step 2 with a changed brief. A real pipeline is a loop with loops inside it, and the design question is not which steps to automate but where the returns are allowed to land.

## Supervising an agent

The hardest part of working with agents is not getting one to run. It is deciding what to let it do, and then checking what it did. Agents produce an enormous amount of work, some of it wrong, and the wrongness concentrates in the steps you were least able to check. Four rules follow.

The first and most useful rule is to give an agent tasks that are cheap to verify. The point is not that the tasks are easy or small, but that the gap between the work of doing and the work of checking is wide. Running a test suite is cheap to verify, because the suite either passes or it does not. Renaming three hundred files to a stated convention is cheap to verify. "Research the current state of the field and tell me what matters" is not, because checking it properly means doing it yourself. When the check costs as much as the task, delegation has bought you nothing but a plausible-sounding answer you now have to argue with.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

fig, ax = plt.subplots(figsize=(10, 4.6))
ax.fill_between([0, 1], 0, 1, color="#f4f1fa", zorder=0)
ax.fill_between([0, 1], -1, 0, color="#fdeef2", zorder=0)
ax.axhline(0, color="#b9b4c4", lw=1.4)
ax.axvline(0, color="#b9b4c4", lw=1.4)

tasks = [
    (0.62, 0.55, "run the test suite\non three hundred files", purple),
    (0.35, 0.30, "rename files to a\nstated convention", purple),
    (0.66, -0.45, "research the field and\ntell me what matters", rose),
    (-0.55, -0.35, "write one sentence\nyou could write yourself", "#666666"),
]
for x, y, label, colour in tasks:
    ax.plot(x, y, "o", color=colour, markersize=11)
    ax.text(x, y - 0.09, label, ha="center", va="top", fontsize=10,
            color=colour, linespacing=1.4)

ax.text(0.97, 0.93, "worth delegating", ha="right", va="top", fontsize=11,
        fontweight="bold", color=purple)
ax.text(0.97, -0.82, "delegation buys you nothing", ha="right", va="bottom",
        fontsize=11, fontweight="bold", color=rose)

ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("how much work the task is to do", labelpad=10)
ax.set_ylabel("how cheap the result is to check", labelpad=10)
ax.text(-0.97, -0.99, "small", ha="left", va="bottom", fontsize=9,
        color="#666666")
ax.text(0.97, -0.99, "large", ha="right", va="bottom", fontsize=9,
        color="#666666")
for side in ax.spines.values():
    side.set_color("#cfc7dd")

plt.tight_layout()
plt.show()

*Figure: The tasks worth handing to an agent are the ones where the work of doing is large and the work of checking is small, and the gap between the two is what you are buying.*

The second rule is to read the trace, not just the result. Skim the steps in order and look for three things: a change of goal that nobody asked for, a failed tool call treated as a success, and an invented fact that kept the loop going. All three are common, all three are visible in the trace, and none of them is visible in the output, which will read confidently either way.

The third rule concerns permissions. An agent is exactly as dangerous as the tools you gave it. Reading files is safe. Writing files is recoverable if you are in [version control](https://en.wikipedia.org/wiki/Version_control), a system that keeps every past state of your files, so that a branch is a copy you can throw away without losing the original. Running arbitrary commands, sending messages under your name, and spending money are three separate powers, and each one deserves a separate decision. Grant the narrowest permission that lets the task finish, work in a copy or a branch, and treat any request to widen the permissions mid-run as a signal to stop and read what has happened so far. Assume, too, that anything the agent reads from the open web may be trying to instruct it.

The fourth rule is budgets, and there are two of them. The one people notice is money. Each step in the loop is a fresh call to a model, and the total for a substantial task on a commercial API is easily ten times the cost of a single conversation. Set a spending cap before you start rather than after your first surprise. The budget people do not notice is time, both the agent's and yours. A loop with no step limit will happily spend an hour going nowhere, and an agent that hands you four hundred lines of output has spent your afternoon rather than saved it.

One more piece is a design decision rather than a safety measure. A human-in-the-loop checkpoint is a point in the pipeline where the agent stops and waits for a person. Put one where the cost of a wrong decision is high and the cost of asking is low. That means before anything irreversible, before anything public, and, in creative work, at every point where the next steps will inherit a choice of taste. In the animated short above, the checkpoint belongs after step 1, because the outline shapes everything downstream and reading three outlines takes two minutes. Checkpoints are where you do your actual job. A pipeline with no checkpoints has not made you more productive, it has made you a reader of finished work you did not choose.

## What agents are not

Four corrections, because the marketing is loud and the reality of a first agent run is usually humbling.

**They are not magical.** How reliable an agent is depends heavily on how long the task is. If each step has a fixed chance of being right, the chance that a whole run is right falls with every step added, and the number of runs you need for one clean result rises with it.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

steps = np.arange(1, 31)
reliabilities = [0.99, 0.95, 0.90]
styles = ["-", "--", ":"]

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.8))

for rate, style in zip(reliabilities, styles):
    left.plot(steps, rate ** steps, color=purple, lw=2.4, ls=style,
              label=f"{rate:.0%} per step")
left.axhline(0.5, color=rose, lw=1.4, ls="--")
left.text(30, 0.53, "half the runs fail", ha="right", fontsize=10, color=rose)
left.set_ylim(0, 1.05)
left.set_xlim(1, 30)
left.set_xlabel("steps in the run")
left.set_ylabel("chance the whole run is right")
left.legend(loc="lower left", frameon=False, title="each step is right")
left.grid(color="#dcd7e4", lw=0.8, ls=":")
left.set_axisbelow(True)

for rate, style in zip(reliabilities, styles):
    right.plot(steps, steps / rate ** steps, color=rose, lw=2.4, ls=style,
               label=f"{rate:.0%} per step")
right.plot(steps, steps, color="#8f8a9c", lw=1.6,
           label="if nothing ever failed")
right.set_yscale("log")
right.set_xlim(1, 30)
right.set_ylim(1, 1200)
right.set_xlabel("steps in the run")
right.set_ylabel("model calls for one run that is right")
right.legend(loc="upper left", frameon=False)
right.grid(color="#dcd7e4", lw=0.8, ls=":")
right.set_axisbelow(True)

for ax in (left, right):
    for side in ("right", "top"):
        ax.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

*Figure: If each step has a fixed chance of being right, the chance of a whole run being right falls with its length, and the calls needed for one run that is right rise with it. The rates shown are illustrative.*

On anything with many steps, expect to supervise, expect to restart, and treat a clean run as a pleasant surprise rather than the baseline.

**They are not cheap at scale.** A single run involves many model calls, and the same work through an agent costs well above asking once. That may still be worth it, but know the order of magnitude before you loop something a hundred times.

**They are not safe by default.** An agent with a browser can buy things, send messages, and sign up to services in your name. Read the permissions before you press go.

**They do not have opinions.** They have outputs. An agent that says your third outline is the strongest has produced a sentence of the kind that usually follows three outlines. Pretending otherwise is a category error, and it is the specific error that makes people delegate the taste and keep the wiring, which is exactly backwards.

:::{note} Dig deeper: how long a task can an agent do?
:class: dropdown
The most careful public measurement of agent capability asks not what fraction of tasks a model can do, but how long a task it can do, measured in the time a skilled human takes [@METR2025]. The finding has two halves. The length of task that models complete with coin-flip reliability has been doubling every few months, which is the half that gets quoted. Within any single model, success falls sharply as tasks get longer, so a system that is dependable on a task of a few minutes is unreliable on one of a few hours.

Look at how the number was produced, because the construction is where the caution lives. The tasks are software engineering and research tasks with clear success criteria, performed in isolation rather than inside somebody's ongoing project, and the human baseline is a timed measurement of skilled people doing the same work. Nothing in the study is a creative task and nothing has an ambiguous finish line. Ask of any agent benchmark: who was timed, on what, and would the task still be scorable if the answer were a matter of judgement.

Two things follow for creative work. The trend is real and the doubling is fast, so a capability judgement made in 2026 will expire, and the honest form of any statement in this chapter is "as of this year". And creative pipelines are the hard case for exactly the reason the study is clean: their success criteria are soft, so the automatic check that made the measurement possible does not exist. In practice the binding constraint on delegating creative work is not the model's task length. It is your supervision capacity, the number of traces you can read carefully in an afternoon, and that is the one thing on the list that does not double every few months.
:::

## Agents in time

Everything so far has assumed an agent whose loop runs in text and whose clock does not matter. If a step takes four seconds instead of one, nothing is lost but patience. Now put the same loop inside a piece of music. The three parts do not change: there are tools, a loop, and a stopping criterion. What changes is that the loop acquires a deadline. A rhythmic response that arrives a few tens of milliseconds late is a different musical event from one that arrives on time, and one that arrives two seconds late is not a response at all. Perception, not the programmer, sets the budget.

This shifts every design decision in the chapter. Planning gets cheap or it does not happen. The stopping criterion stops being "the goal is met" and becomes "the next beat has arrived", so the agent must always have something to play, including when it has not worked out what. And supervision cannot be a checkpoint, because there is nowhere to pause. You supervise a real-time agent the way a bandleader supervises a player, by changing what you do and hearing what comes back.

[Dr. Squiggles](https://www.uio.no/ritmo/english/projects/dr-squiggles/) is the clearest example in this book. It is a set of small autonomous musical robots built at RITMO between 2020 and 2022 [@DrSquiggles]. Each one listens through a microphone, estimates where the beat is with a real-time beat-tracking library, and taps a rhythmic response with its limbs. The response is generated rather than played back, so what comes out depends on what the robot has just heard. Read that as an agent and the three parts line up. The tools are a microphone and a set of tapping limbs; the loop is listen, estimate, respond; and the stopping criterion is the one thing that is different, because there is not one. The loop runs continuously, and its real constraint is that each turn must finish before the next beat, a matter of tens of milliseconds (a fiftieth of a second) rather than seconds.

Put several of these robots in a room and each listens to the others as well as to any human present, so every unit is both a source and a follower. Krzyżaniak found that the timing behaviour of such a group is not a property of any single robot's algorithm but of the coupling between them [@Krzyzaniak2021]. Small changes in how strongly a unit adjusts to what it hears decide whether the group converges on a shared pulse, drifts, or oscillates. That is a multi-agent system in the strict sense, and it shows what the term costs: with several loops running at once the behaviour is emergent, a polite word for not directly designed, and supervision cannot mean approving each step. It also shows that a loop can be very simple and still produce something musically interesting. What it needs is not a large model in the middle but a clock it can meet.

The pipeline model of this chapter, a chain of steps with checks between them, is therefore only one shape an agent can take. The other is a participant, a system always in the middle of the loop, judged by whether it fits rather than by whether it is correct. Chapter [10](body.ipynb) takes up that second shape, with bodies, sensors, and rooms.

## Writing the brief

The centre of gravity of generative AI is moving from "produce one artefact from one prompt" to "complete one task using many tools over time". The work is still yours, but you do it as a director rather than as a maker, and the value sits in the brief, the supervision, and the taste, which are the three things the loop cannot supply.

For students this collapses into one piece of practical advice: become very good at writing briefs. A good brief states the goal, the constraints, and what finished looks like. It says what the result is for and who will see it, so that the system has a criterion rather than a topic. It names what must not happen. And it is honest about which decisions are open, because a brief that pretends to be open when you have already decided will produce three variants you were never going to use.

That skill is not new and it is not technical; it is what a commissioning editor, an art director, and a supervisor have always done. What is new is that a person who can write a good brief, and read a trace carefully enough to know when it went wrong, can now ship work that used to require a team.

:::{admonition} Research spotlight
:class: spotlight
The [Musical Gestures Toolbox for Python](https://github.com/fourMs/MGT-python) appeared in chapter [5](images.ipynb) as an instrument that measures rather than generates, and as a way of seeing a whole clip at once [@MGTpython]. This week it is neither. It is a codebase, and it is the right size to read.

Three properties make it unusual as a piece of software. It is small enough that one person can hold most of it in their head. It installs and runs on a laptop without a cluster, a licence, or a dataset you do not have. And every line of it is public. Most research software fails at least one of those tests, which is why so few students ever open the source of anything a scientist wrote. This one you can clone, run on your own footage in an afternoon, and then look at how the function you just called works.

Do that at least once with an assistant beside you, because reading a real library is a different exercise from reading a tutorial. Point the assistant at one module and ask what a function returns, why a default was chosen, what happens with an unusual input. Tutorial code is written to be understood, so it is simplified until nothing is left but the idea. Research code is written to answer a question, so the researchers' assumptions about the recordings they had are sitting in the defaults. A threshold that separates motion from noise, a frame rate, a colour map, an assumption that the camera did not move: each was set for a particular kind of recording in a particular lab. Each is a decision rather than a fact. Finding one of them and working out why it was chosen teaches you more about reading code than any exercise this chapter could set.

Then extend it. Add a function that does something the toolbox does not, run it on material you recorded yourself, and ask whether the number it gives you is one you would defend to the person who wrote the module you built on. That is a more demanding standard than "it runs", and it is the standard research code is held to.
:::

## This week's lab: Explore, Reflect, Create

Three movements. Read and break a sketch with an assistant, and watch a loop run. Argue with a partner about what you would hand to a loop. Then build a sketch, or run one slice of your final piece as a supervised pipeline.

**A3 is due this week.** *Multimodal mini-piece* was set in week 6 and is handed in now, so keep time clear for it before you start; nothing you make in this lab replaces it.

**The proposal for your final piece starts this week.** It is a checkpoint, not an assessed assignment: one to two pages plus a feasibility sketch, due in week 9, naming the shape of the final piece and how a reader will meet it in your exposition. The final piece is made in weeks 8 to 10, and the exposition drafts go to peer review in week 10, so the pipeline you draw today should be the one you are going to build.

### Explore (about 30 min)

**One feature, two assistants, then one loop.**

1. Open the [p5.js web editor](https://editor.p5js.org/), paste in the hello-world sketch from this chapter, and confirm it runs.
2. Choose one small feature, such as "make the background fade slowly instead of clearing", and prompt two different assistants for it with the same wording. Any two of the four patterns will do, or two chat assistants, one of them a local model if you have one running.
3. Run both. Note which is idiomatic p5.js, which handled the first frame correctly, and which explained itself when you asked "why did you write it this way?" Then ask the weaker one to critique the stronger one's version, and note whether the critique is useful or merely polite.
4. Open the [Agent loop simulator](https://fourms.github.io/Creative-AI/apps/agent-loop/). Give it a toy task, such as finding the cheapest train to Bergen next Tuesday, and step the loop through three times. Make one observation noisy, so that what comes back is incomplete or wrong, and watch what the next plan does with it. That is the moment you are looking for: where the loop commits to a wrong step and everything afterwards is downstream of it.
5. If you have an agentic assistant, give it one bounded task with a cheap check, such as making a small script pass a test you wrote first. Run it once without intervening, then read the whole trace in order and mark every step as sound, wasteful, or wrong. Find the first wrong step, which is rarely the one where the failure became visible. The research mode of a chat assistant runs the same loop and prints the same kind of trace if no agent tool is available.

Two short paragraphs in your log, plus both versions of the code and the trace, are enough.

### Reflect (about 15 min)

Work in pairs. This is a discussion, not a writing block.

1. Take the longest function either assistant produced and explain it to your partner, out loud, line by line, without looking at any explanation the assistant offered. Your partner interrupts at every line you gloss over. Then swap.
2. Show each other the trace and the first wrong step, and decide together whether a checkpoint could have caught it.
3. Take your final pieces in turn. List the steps you would delegate and the steps you would keep, and say for each delegated step what the cheap check is. If you cannot name the check, move the step back into the keep column. Name one permission you would never grant an agent working on your piece, and push each other for the specific harm.
4. Close by each stating aloud which path you are taking in the Create phase, and one sentence on your final piece, including what a reader of your exposition would meet first.

### Create (about 45 min)

Take one of the paths below and publish what you make. Paths A and B are built in the [p5.js web editor](https://editor.p5js.org/), path C is live coded in the browser, and path D is your final piece, drawn as a pipeline and run one slice at a time.

**Path A: a mouse-reactive sketch.** Build a sketch with an assistant that meets at least two of: colour that changes with position or speed; shapes that leave trails; sound on click; a canvas that responds to window size. Iterate at least three times, keeping the versions, and after each iteration write one line in your log saying what you asked for and what you got. Save the sketch publicly and put the link in your log with a sentence on what you would fix next.

**Path B: an evolving sketch.** Write one drawing function that takes a small genome, a handful of numbers controlling colour, angle, count, and thickness, and draws a single figure from it. Draw nine of them in a grid, each the same parent genome with a small random change, and make a click on a tile promote that genome to parent and redraw the grid. Run at least four generations, keeping a screenshot of each. Save the sketch publicly and put the link in your log with one sentence on what you found yourself selecting for.

**Path C: a live-coded piece.** Open [Glicol](https://glicol.org/) or [Strudel](https://strudel.cc/) and work from a built-in example until you have a sound worth keeping. Build a piece of about ninety seconds that you can perform from a nearly empty editor, and rehearse it three times. Use an assistant only between runs, never during one, and note what you asked for and whether you kept it. Record the screen and the sound together, publish the recording, and put the link in your log with one sentence on what an audience could see that they could not hear.

**Path D: one slice of your final piece.** Draw the pipeline, every step from nothing to finished piece, with the returns drawn in. Annotate each step with the model or tool, what goes in, what comes out, how you would verify the output, what happens when it fails, and whether there is a human checkpoint. Mark at least one checkpoint deliberately, before something irreversible or before a choice of taste that everything downstream inherits. Shade the steps you will delegate differently from the steps you will keep; that split is the finding. Then run one delegated step for real, keep the trace, and write fifteen lines on what the slice cost, what you had to check, and whether the delegation was worth it. An honest no is a result. Put the diagram, the trace, and the notes in your portfolio; they are the first draft of your proposal.

**Optional, for the commons.** Look up the tool you built with, or the artist whose work inspired your sketch, on [Wikidata](https://www.wikidata.org/), the structured sister project of Wikipedia. If the item lacks a statement you can source, such as the developer, the licence, or the release date, add it by hand with a reference. Wikidata is released under CC0, so every statement you add is free for anyone to reuse, including the companies that train models on it.

At home, write this week's entry in your practice log using the [practice log template](templates/practice-log.md).

## A critical look: does AI make beginners better programmers?

**The claim.** Coding assistants remove the pointless part of learning to program. Beginners no longer lose weeks to a missing semicolon, they get to interesting problems in their first session, and everyone who used to be locked out by syntax can now build things. The tool is a ladder into the discipline.

**The evidence.** Two studies from 2023 are the ones to know, and together they say something more complicated than either says alone. In a controlled study with sixty-nine novice learners aged ten to seventeen, the group given a code generator during a set of introductory programming tasks completed significantly more of them and wrote more correct code than the control group. On a retention test a week later they did not perform worse [@Kazemitabaar2023]. That result is real, and it is narrower than the claim. The tasks were short and introductory, the follow-up was one week, and the comparison was between having the tool and not having it rather than between two ways of teaching. An observational think-aloud study of university students working with a tab-completion assistant supplies the other half of the picture. It found beginners struggling to evaluate suggestions they had not asked for, accepting code they could not explain, and getting stuck in ways not reported in the earlier literature on novice programmers. One characteristic pattern was confidence unsupported by understanding [@Prather2023]. More tasks were completed, then, and a real question remains about what was completed *in* the student.

**The method.** The two are built differently, and the difference decides what each can license. The first is a controlled experiment with a control group and a delayed test, so it can establish that the tool caused the difference in what was completed. It cannot say what those learners will understand in five years, because it did not run for five years. The second is an observational study in which students narrated their work aloud while researchers watched, with no comparison arm. It can describe behaviour nobody had thought to look for, which is exactly what it did, but it cannot say how common that behaviour is, because nothing was counted against a baseline. Both are small, short, and run in a single setting, which is the ordinary condition of research in this area. Read them as complementary: one says the tool changes what gets finished, the other says something about what happens in the person while it does. Neither speaks to what you will be able to do in ten years, and the published work says so plainly. The claims made on their behalf, in course descriptions and in marketing, usually do not.

**The limits.** The honest version is that assistants help beginners *complete* more, that the short-term learning evidence is mixed rather than negative, and that the long-term question is open. Whether a generation of programmers who learned this way can debug a system nobody understands is not something a one-week retention test can answer. Both studies also predate the agentic pattern, in which far more code passes without being read, so the question they leave open has grown since they were run. And "better programmer" is not one thing. Producing working code, reading unfamiliar code, choosing an approach, and debugging under pressure are separable skills, and there is no reason to expect a tool to move them together. The available evidence is mostly about the first.

This course is arranged around that uncertainty. It grades the practice log and the explaining alongside the artefact, and it asks you to account for what you shipped rather than only to ship it. If the deskilling worry turns out to be unfounded, you will have lost very little. If it turns out to be right, the reading is the part that will still be yours.

```{admonition} Chapter summary
:class: tip
Coding assistants come in four patterns, tab completion, chat in the editor, inline edit, and agentic, and they differ mainly in how much unreviewed code each one adds per interaction. Because a model can write far more code than you can, the binding constraint moved from writing to reading, which makes supervision a reading skill, and explain-back is the habit that builds it. p5.js is a good place to practise, since a sketch runs in the browser, breaks visibly, and can be taken apart a line at a time. The backwards loop in the trailing-circles example is the kind of detail that separates code that runs from code you understand. Talking well to an assistant means naming the stack, showing the smallest failing example, asking for explanations, failure cases, and tests before fixes, changing one thing at a time, and verifying by running. Code is also a rule-based and evolutionary medium in its own right. An L-system, a cellular automaton, or a nine-tile interactive-evolution loop gives you material an assistant can build but cannot judge for you, and live coding takes the same medium on stage. A tiny AI-powered web tool needs four parts: a page, a model call, a key that never reaches the browser, and somewhere to host it. The agentic assistant is the general case of all this: a model with tools, a loop of act and observe, and a stopping criterion, made workable from 2023 onwards by models that can notice a failed step and try again. A creative pipeline maps onto that shape, and what gets automated is the wiring between the creative decisions rather than the decisions. Supervision means preferring tasks that are cheap to verify, reading the trace rather than only the result, granting the narrowest permission, capping both budgets, and placing human checkpoints where a wrong decision is expensive and asking is cheap. Agents are not magical, not cheap, not safe by default, and not holders of opinions. When the loop runs in musical time, the deadline replaces the stopping criterion and supervision becomes performance, which is the bridge to chapter [10](body.ipynb). The value has moved to the brief, the supervision, and the taste. And the evidence on whether assistants make better programmers says that beginners complete more, that short-term retention is mixed rather than damaged, and that the long-term question is open. That is why this course grades reading and explaining and not only shipping.
```

```{admonition} Questions
:class: question
1. Name the four assistant patterns and order them by how much unreviewed code each one can add to your project in a single interaction. Which did you use this week, and did the ordering match your experience?
2. Why does this chapter argue that reading code matters more than writing it now? Give the argument in terms of where the bottleneck sits, and describe one bug an assistant introduced into your sketch this week that no error message would have shown you.
3. Name the three parts of the minimum agent architecture, and say which of them the Dr. Squiggles robots do not have in the usual form, and why.
4. Why is *cheap to verify* a better criterion for delegation than *easy* or *small*? Give one step from your final piece that satisfies it and one that does not, and name a permission you would never grant an agent working on that piece, with the specific harm.
5. What does a good brief contain? Write one for a step of your pipeline, and say which sentence in it gives the system a criterion rather than a topic.
```

:::{seealso} Further reading
- **Processing Foundation, *p5.js Tutorials***: the official tutorials and reference, well designed and free; start with the interactive examples rather than the reference [@P5js]. [p5js.org/tutorials](https://p5js.org/tutorials/)
- **Daniel Shiffman, *The Nature of Code* (2024)**: a free book on generative graphics, motion, and simulation in p5.js; the best next step after this chapter's lab [@Shiffman2024NatureOfCode]. [natureofcode.com](https://natureofcode.com/)
- **Daniel Shiffman, *The Coding Train***: long-form creative-coding lectures on video, for when reading is not the format you want [@CodingTrain]. [youtube.com/@TheCodingTrain](https://www.youtube.com/@TheCodingTrain)
- **Karl Sims, *Artificial evolution for computer graphics* (1991)**: the paper that put interactive evolution on the SIGGRAPH stage, and the images still hold up [@Sims1991]. [doi.org/10.1145/127719.122752](https://doi.org/10.1145/127719.122752)
- **Prusinkiewicz and Lindenmayer, *The Algorithmic Beauty of Plants* (1990)**: the L-system reference, and the best argument anywhere for rules as a drawing medium [@Prusinkiewicz1990]. [doi.org/10.1007/978-1-4613-8476-2](https://doi.org/10.1007/978-1-4613-8476-2)
- **Blackwell, Cocker, Cox, McLean and Magnusson, *Live Coding: A User's Manual* (2022)**: the field's own account of live coding, its history, and its politics, free to read [@Blackwell2022LiveCoding]. [doi.org/10.7551/mitpress/13770.001.0001](https://doi.org/10.7551/mitpress/13770.001.0001)
- **Lan and Jensenius, *Collaborative Live Coding With Glicol Music Programming Language* (2021)**: a documented Web Audio Conference performance in which workshop participants joined over the web and each wrote a loop [@Lan2021Glicol]. [doi.org/10.5281/zenodo.6539832](https://doi.org/10.5281/zenodo.6539832)
- **Kazemitabaar et al., *Studying the effect of AI code generators on supporting novice learners in introductory programming* (2023)**: the controlled study behind the first half of the critical look [@Kazemitabaar2023]. [doi.org/10.1145/3544548.3580919](https://doi.org/10.1145/3544548.3580919)
- **Prather et al., *"It's weird that it knows what I want": usability and interactions with Copilot for novice programmers* (2023)**: the observational study behind the second half; read the descriptions of where students got stuck [@Prather2023]. [doi.org/10.1145/3617367](https://doi.org/10.1145/3617367)
- **Model Evaluation and Threat Research, *Measuring AI ability to complete long tasks* (2025)**: the study behind the Dig deeper note on task length; read the task-length figure and then read how the human baselines were timed [@METR2025]. [arxiv.org/abs/2503.14499](https://arxiv.org/abs/2503.14499)
- **Anthropic, *Claude Code* documentation**: a readable and opinionated account of how a terminal coding agent is meant to be supervised, including permissions and stopping [@ClaudeCode]. [docs.anthropic.com/claude-code](https://docs.anthropic.com/claude-code)
- **Cursor documentation**: the chat and inline-edit patterns documented from the builders' side, worth reading rather than only using [@CursorDocs]. [cursor.com/docs](https://cursor.com/docs)
- **Andrej Karpathy, *Software Is Changing (Again)* (2025)**: the talk that names Software 3.0, programming in natural language, which is the argument behind treating a brief as an interface [@Karpathy2025Software]. [youtube.com/watch?v=LCEmiRjPEtQ](https://www.youtube.com/watch?v=LCEmiRjPEtQ)
- **Benj Edwards, *10 Things I Learned from Burning Myself Out with AI Coding Agents* (2026)**: a working programmer on what a day of supervising agents does to the person supervising [@Edwards2026Burnout]. [arstechnica.com](https://arstechnica.com/information-technology/2026/01/10-things-i-learned-from-burning-myself-out-with-ai-coding-agents/)
- **Hugging Face Spaces**: hundreds of one-page AI demos with their source visible, which is the fastest way to see what a tiny AI-powered tool looks like inside [@HuggingFaceSpaces]. [huggingface.co/spaces](https://huggingface.co/spaces)
:::

:::{tip} Explore interactively
- [p5.js web editor](https://editor.p5js.org/): write, run, and publish sketches in the browser with nothing installed. Paths A and B of this week's lab happen here.
- [Agent loop simulator](https://fourms.github.io/Creative-AI/apps/agent-loop/): step a loop through by hand, give it tools, change the stopping criterion, and watch where a run ends when the first wrong step goes unnoticed.
- [Evolve a picture](https://fourms.github.io/Creative-AI/apps/evolve/): the nine-tile breeding loop from Path B, already built, if you would rather play with one before writing one.
- [Glicol](https://glicol.org/) (external): a graph-oriented live coding language that runs in a browser tab. Load an example, press play, and change a number while it sounds.
- [synth.is](https://synth.is/) (external): sounds bred by evolutionary search rather than written by hand, browsable as a family tree.
:::